In [174]:
%%writefile config.py
"""Configuration file for Hybrid-Stem Swin+UNet Model"""
import torch

class Config:
    # ==================== PATHS ====================
    TRAIN_AUTHENTIC_DIR = "/kaggle/input/sci-doc-forgery-data/supplemental_images/synthetic_forgery_dataset/authentic"
    TRAIN_FORGED_DIR = "/kaggle/input/sci-doc-forgery-data/supplemental_images/synthetic_forgery_dataset/forged/images"
    TRAIN_MASK_DIR = "/kaggle/input/sci-doc-forgery-data/supplemental_images/synthetic_forgery_dataset/masks"
    
    TEST_AUTHENTIC_DIR = "/kaggle/input/sci-doc-forgery-data/recodai-luc-scientific-image-forgery-detection/recodai-luc-scientific-image-forgery-detection/train_images/authentic"
    TEST_FORGED_DIR = "/kaggle/input/sci-doc-forgery-data/recodai-luc-scientific-image-forgery-detection/recodai-luc-scientific-image-forgery-detection/train_images/forged"
    TEST_MASK_DIR = "/kaggle/input/sci-doc-forgery-data/recodai-luc-scientific-image-forgery-detection/recodai-luc-scientific-image-forgery-detection/train_masks"
    
    HQ_AUTHENTIC_DIR = "/kaggle/input/sci-doc-forgery-data/recodai-luc-scientific-image-forgery-detection"
    HQ_FORGED_DIR = "/kaggle/input/sci-doc-forgery-data/supplemental_images/supplemental_images"
    HQ_MASK_DIR = "/kaggle/input/sci-doc-forgery-data/supplemental_images/supplemental_masks"
    
    OUTPUT_DIR = "/kaggle/working/outputs"
    CHECKPOINT_DIR = "/kaggle/working/checkpoints"
    
    # ==================== MODEL ====================
    # HUGE UPDATE: 896px Input
    INPUT_SIZE = 896
    MASK_SIZE = 224
    INPUT_CHANNELS = 3
    NUM_INSTANCES = 6
    
    SWIN_MODEL = "swin_small_patch4_window7_224"
    EMBEDDING_DIM = 128
    
    DECODER_CHANNELS = [512, 256, 128, 64, 32]
    
    # ==================== TRAINING ====================
    NUM_EPOCHS = 45
    BATCH_SIZE = 26
    GRADIENT_ACCUMULATION_STEPS = 10
    
    ENCODER_FREEZE_EPOCHS = 1
    
    LEARNING_RATE = 4e-4
    WEIGHT_DECAY = 1e-4
    OPTIMIZER = "AdamW"
    
    T_MAX = NUM_EPOCHS
    ETA_MIN = 1e-6
    
    # Mixed Precision
    USE_AMP = True
    
    # ==================== LOSS WEIGHTS ====================
    LOSS_WEIGHT_DICE = 1.0
    LOSS_WEIGHT_SEGMENTATION_FOCAL = 2.0
    LOSS_WEIGHT_ACTIVE = 4.0
    LOSS_WEIGHT_CONTRASTIVE = 1.0
    
    FOCAL_ALPHA = 0.9
    FOCAL_GAMMA = 1.2
    SEG_FOCAL_ALPHA = 0.9
    SEG_FOCAL_GAMMA = 2.0
    
    # Contrastive Config
    CONTRASTIVE_TEMPERATURE = 0.07
    CONTRASTIVE_SAMPLES_BACKGROUND = 256
    CONTRASTIVE_SAMPLES_PER_INSTANCE = 64
    CONTRASTIVE_MARGIN_BG_FG = 1.5   
    CONTRASTIVE_MARGIN_FG_FG = 1.0   
    CONTRAST_WEIGHT_BG_FG = 2.0      
    CONTRAST_WEIGHT_FG_FG = 1.2      
    
    # ==================== AUGMENTATION ====================
    AUG_ROTATION_LIMIT = 30 
    AUG_PROB_FLIP = 0.5
    AUG_PROB_ROTATE = 0.5
    
    AUG_BRIGHTNESS = 0.25
    AUG_CONTRAST = 0.15
    AUG_SATURATION = 0.15
    AUG_HUE = 0.05
    AUG_PROB_COLOR = 0.5
    
    AUG_JPEG_QUALITY_RANGE = (80, 100) # Slightly more aggressive
    AUG_PROB_JPEG = 0.4
    
    AUG_BLUR_LIMIT = (3, 5) 
    AUG_PROB_BLUR = 0.3
    
    AUG_NOISE_VAR_LIMIT = (5.0, 15.0) 
    AUG_PROB_NOISE = 0.3
    
    # ==================== INFERENCE ====================
    ACTIVE_CHANNEL_THRESHOLD = 0.5
    MASK_THRESHOLD = 0.5
    
    # ==================== VALIDATION ====================
    VAL_SAMPLE_RATIO = 0.5
    VAL_AUTHENTIC_RATIO = 0.3
    VAL_FORGED_RATIO = 0.7
    VAL_FREQUENCY = 3

    # ==================== TUNING ====================
    # Tuning on Training Data
    TUNING_SAMPLE_RATIO = 0.15        # 15% of total data
    TUNING_AUTHENTIC_RATIO = 0.30     # 30% Authentic
    TUNING_FORGED_RATIO = 0.70        # 70% Forged
    
    # Grid Search Range
    TEST_ACTIVE_THRESHOLDS = [0.2, 0.3, 0.4, 0.5, 0.6, 0.7]
    TEST_MASK_THRESHOLDS = [0.3, 0.4, 0.5, 0.6]
    
    # ==================== DISTRIBUTED ====================
    SAVE_TOP_WORST = 100
    WORLD_SIZE = 2
    BACKEND = "nccl"
    
    # ==================== SMOKE TEST ====================
    SMOKE_TEST = False
    SMOKE_TEST_SAMPLES = 30
    SMOKE_TEST_EPOCHS = 30
    SMOKE_TEST_BATCH_SIZE = 4
    SMOKE_TEST_FREEZE_EPOCHS = 10
    SMOKE_TEST_GRAD_ACCUM = 1
    
    @classmethod
    def set_smoke_test(cls, enable=True):
        cls.SMOKE_TEST = enable
        if enable:
            cls.NUM_EPOCHS = cls.SMOKE_TEST_EPOCHS
            cls.ENCODER_FREEZE_EPOCHS = cls.SMOKE_TEST_FREEZE_EPOCHS
            cls.T_MAX = cls.SMOKE_TEST_EPOCHS
            cls.BATCH_SIZE = cls.SMOKE_TEST_BATCH_SIZE
            cls.GRADIENT_ACCUMULATION_STEPS = cls.SMOKE_TEST_GRAD_ACCUM
            print(f"🔥 SMOKE TEST MODE ENABLED")

    @classmethod
    def print_config(cls):
        print("=" * 60)
        print("HYBRID-STEM MODEL CONFIGURATION")
        print("=" * 60)
        for key, value in cls.__dict__.items():
            if not key.startswith('_') and not callable(value):
                print(f"{key}: {value}")
        print("=" * 60)

Overwriting config.py


In [175]:
%%writefile utils.py
"""
Utility functions: RLE encoding/decoding and OF1 metric
Save as: utils.py
"""

import json
import numba
import numpy as np
from numba import types
import numpy.typing as npt
import pandas as pd
import scipy.optimize


class ParticipantVisibleError(Exception):
    pass


@numba.jit(nopython=True)
def _rle_encode_jit(x: npt.NDArray, fg_val: int = 1) -> list[int]:
    """Numba-jitted RLE encoder."""
    dots = np.where(x.T.flatten() == fg_val)[0]
    run_lengths = []
    prev = -2
    for b in dots:
        if b > prev + 1:
            run_lengths.extend((b + 1, 0))
        run_lengths[-1] += 1
        prev = b
    return run_lengths


def rle_encode(masks: list[npt.NDArray], fg_val: int = 1) -> str:
    """
    Encode masks to RLE format.
    Args:
        masks: list of numpy array of shape (height, width), 1 - mask, 0 - background
    Returns: run length encodings as a string, with each RLE JSON-encoded and separated by a semicolon.
    """
    return ';'.join([json.dumps(_rle_encode_jit(x, fg_val)) for x in masks])


@numba.njit
def _rle_decode_jit(mask_rle: npt.NDArray, height: int, width: int) -> npt.NDArray:
    """
    Decode RLE to mask.
    s: numpy array of run-length encoding pairs (start, length)
    shape: (height, width) of array to return
    Returns numpy array, 1 - mask, 0 - background
    """
    if len(mask_rle) % 2 != 0:
        raise ValueError('One or more rows has an odd number of values.')

    starts, lengths = mask_rle[0::2], mask_rle[1::2]
    starts -= 1
    ends = starts + lengths
    for i in range(len(starts) - 1):
        if ends[i] > starts[i + 1]:
            raise ValueError('Pixels must not be overlapping.')
    img = np.zeros(height * width, dtype=np.bool_)
    for lo, hi in zip(starts, ends):
        img[lo:hi] = 1
    return img


def rle_decode(mask_rle: str, shape: tuple[int, int]) -> npt.NDArray:
    """
    Decode RLE string to mask.
    mask_rle: run-length as string formatted (start length)
              empty predictions need to be encoded with '-'
    shape: (height, width) of array to return
    Returns numpy array, 1 - mask, 0 - background
    """
    mask_rle = json.loads(mask_rle)
    mask_rle = np.asarray(mask_rle, dtype=np.int32)
    starts = mask_rle[0::2]
    if sorted(starts) != list(starts):
        raise ParticipantVisibleError('Submitted values must be in ascending order.')
    try:
        return _rle_decode_jit(mask_rle, shape[0], shape[1]).reshape(shape, order='F')
    except ValueError as e:
        raise ParticipantVisibleError(str(e)) from e


def calculate_f1_score(pred_mask: npt.NDArray, gt_mask: npt.NDArray):
    """Calculate F1 score between two binary masks."""
    pred_flat = pred_mask.flatten()
    gt_flat = gt_mask.flatten()

    tp = np.sum((pred_flat == 1) & (gt_flat == 1))
    fp = np.sum((pred_flat == 1) & (gt_flat == 0))
    fn = np.sum((pred_flat == 0) & (gt_flat == 1))

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0

    if (precision + recall) > 0:
        return 2 * (precision * recall) / (precision + recall)
    else:
        return 0


def calculate_f1_matrix(pred_masks: list[npt.NDArray], gt_masks: list[npt.NDArray]):
    """
    Calculate F1 matrix for all pairs of predicted and ground truth masks.
    """
    num_instances_pred = len(pred_masks)
    num_instances_gt = len(gt_masks)
    f1_matrix = np.zeros((num_instances_pred, num_instances_gt))

    for i in range(num_instances_pred):
        for j in range(num_instances_gt):
            pred_flat = pred_masks[i].flatten()
            gt_flat = gt_masks[j].flatten()
            f1_matrix[i, j] = calculate_f1_score(pred_mask=pred_flat, gt_mask=gt_flat)

    if f1_matrix.shape[0] < len(gt_masks):
        # Add rows of zeros if fewer predictions than ground truth
        f1_matrix = np.vstack((f1_matrix, np.zeros((len(gt_masks) - len(f1_matrix), num_instances_gt))))

    return f1_matrix


def oF1_score(pred_masks: list[npt.NDArray], gt_masks: list[npt.NDArray]):
    """
    Calculate optimal F1 score using Hungarian algorithm.
    """
    f1_matrix = calculate_f1_matrix(pred_masks, gt_masks)
    row_ind, col_ind = scipy.optimize.linear_sum_assignment(-f1_matrix)
    excess_predictions_penalty = len(gt_masks) / max(len(pred_masks), len(gt_masks))
    return np.mean(f1_matrix[row_ind, col_ind]) * excess_predictions_penalty


def evaluate_single_image(label_rles: str, prediction_rles: str, shape_str: str) -> float:
    """Evaluate single image with RLE inputs."""
    shape = json.loads(shape_str)
    label_rles = [rle_decode(x, shape=shape) for x in label_rles.split(';')]
    prediction_rles = [rle_decode(x, shape=shape) for x in prediction_rles.split(';')]
    return oF1_score(prediction_rles, label_rles)


def score(solution: pd.DataFrame, submission: pd.DataFrame, row_id_column_name: str) -> float:
    """
    Calculate overall score for entire dataset.
    """
    df = solution
    df = df.rename(columns={'annotation': 'label'})

    df['prediction'] = submission['annotation']
    # Check for correct 'authentic' label
    authentic_indices = (df['label'] == 'authentic') | (df['prediction'] == 'authentic')
    df['image_score'] = ((df['label'] == df['prediction']) & authentic_indices).astype(float)

    df.loc[~authentic_indices, 'image_score'] = df.loc[~authentic_indices].apply(
        lambda row: evaluate_single_image(row['label'], row['prediction'], row['shape']), axis=1
    )
    return float(np.mean(df['image_score']))

Overwriting utils.py


In [176]:
%%writefile dataset.py
"""Dataset with Image Name Restored"""
import os
import random
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from torch.utils.data.distributed import DistributedSampler
import albumentations as A
import cv2

class ForgeryDataset(Dataset):
    def __init__(self, authentic_dir, forged_dir, mask_dir, transform=None, config=None):
        self.authentic_dir = authentic_dir
        self.forged_dir = forged_dir
        self.mask_dir = mask_dir
        self.transform = transform
        self.config = config
        
        self.samples = []
        if os.path.exists(authentic_dir):
            auth_imgs = sorted([f for f in os.listdir(authentic_dir) if f.endswith(('.jpg', '.png', '.jpeg'))])
            for img in auth_imgs:
                self.samples.append({'image': img, 'is_forged': False, 'dir': authentic_dir})
                
        if os.path.exists(forged_dir):
            forg_imgs = sorted([f for f in os.listdir(forged_dir) if f.endswith(('.jpg', '.png', '.jpeg'))])
            for img in forg_imgs:
                self.samples.append({'image': img, 'is_forged': True, 'dir': forged_dir})
        
        print(f"📦 Total samples: {len(self.samples)}")
        
        if config and config.SMOKE_TEST:
            random.shuffle(self.samples)
            self.samples = self.samples[:config.SMOKE_TEST_SAMPLES]
            print(f"🔥 SMOKE TEST: Sampled {len(self.samples)} images")
    
    def __len__(self):
        return len(self.samples)
    
    def letterbox_processed(self, image, masks, img_size, mask_size):
        h, w = image.shape[:2]
        scale = min(img_size / h, img_size / w)
        new_h, new_w = int(h * scale), int(w * scale)
        
        resized_image = cv2.resize(image, (new_w, new_h), interpolation=cv2.INTER_LINEAR)
        padded_image = np.zeros((img_size, img_size, 3), dtype=np.uint8)
        y_off = (img_size - new_h) // 2
        x_off = (img_size - new_w) // 2
        padded_image[y_off:y_off+new_h, x_off:x_off+new_w] = resized_image
        
        scale_mask = min(mask_size / h, mask_size / w)
        new_h_mask, new_w_mask = int(h * scale_mask), int(w * scale_mask)
        
        padded_masks = []
        for mask in masks:
            m_resized = cv2.resize(mask.astype(np.uint8), (new_w_mask, new_h_mask), interpolation=cv2.INTER_NEAREST)
            p_mask = np.zeros((mask_size, mask_size), dtype=np.uint8)
            y_off_m = (mask_size - new_h_mask) // 2
            x_off_m = (mask_size - new_w_mask) // 2
            p_mask[y_off_m:y_off_m+new_h_mask, x_off_m:x_off_m+new_w_mask] = m_resized
            padded_masks.append(p_mask)
            
        return padded_image, padded_masks
    
    def __getitem__(self, idx):
        try:
            sample = self.samples[idx]
            img_path = os.path.join(sample['dir'], sample['image'])
            
            image = cv2.imread(img_path)
            if image is None: raise ValueError("Image not found")
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
            
            masks = []
            if sample['is_forged']:
                mask_path = os.path.join(self.mask_dir, os.path.splitext(sample['image'])[0] + '.npy')
                if os.path.exists(mask_path):
                    mask_data = np.load(mask_path, allow_pickle=True)
                    h, w = image.shape[:2]
                    if mask_data.ndim == 2:
                        if mask_data.shape != (h, w):
                            mask_data = cv2.resize(mask_data.astype(np.uint8), (w, h), interpolation=cv2.INTER_NEAREST)
                        masks.append(mask_data)
                    elif mask_data.ndim == 3:
                        for i in range(mask_data.shape[0]):
                             if np.sum(mask_data[i]) > 0:
                                m = mask_data[i]
                                if m.shape != (h, w):
                                    m = cv2.resize(m.astype(np.uint8), (w, h), interpolation=cv2.INTER_NEAREST)
                                masks.append(m)

            img_size = self.config.INPUT_SIZE
            mask_size = self.config.MASK_SIZE
            
            image, masks = self.letterbox_processed(image, masks, img_size, img_size)
            
            if self.transform:
                if masks:
                    aug = self.transform(image=image, masks=masks)
                    image, masks = aug['image'], aug['masks']
                else:
                    image = self.transform(image=image)['image']
            
            final_masks = []
            for m in masks:
                m_small = cv2.resize(m.astype(np.uint8), (mask_size, mask_size), interpolation=cv2.INTER_NEAREST)
                final_masks.append(m_small)
            masks = final_masks
            
            while len(masks) < self.config.NUM_INSTANCES:
                masks.append(np.zeros((mask_size, mask_size), dtype=np.float32))
            masks = masks[:self.config.NUM_INSTANCES]
            
            image = torch.from_numpy(image).permute(2, 0, 1).float() / 255.0
            masks = torch.stack([torch.from_numpy(m.astype(np.float32)) for m in masks])
            
            return {
                'input': image,
                'masks': masks,
                'is_forged': torch.tensor(sample['is_forged'], dtype=torch.float32),
                'num_instances': torch.tensor(len([m for m in masks if m.sum() > 0]), dtype=torch.long),
                'image_name': sample['image'] # <--- RESTORED THIS KEY
            }
        except Exception:
            return self.__getitem__(random.randint(0, len(self.samples)-1))

def get_train_transform(config):
    return A.Compose([
        A.HorizontalFlip(p=config.AUG_PROB_FLIP),
        A.VerticalFlip(p=config.AUG_PROB_FLIP),
        A.Rotate(limit=config.AUG_ROTATION_LIMIT, p=config.AUG_PROB_ROTATE),
        A.GaussianBlur(blur_limit=config.AUG_BLUR_LIMIT, p=config.AUG_PROB_BLUR),
        A.GaussNoise(var_limit=config.AUG_NOISE_VAR_LIMIT, p=config.AUG_PROB_NOISE),
        A.ImageCompression(quality_range=config.AUG_JPEG_QUALITY_RANGE, p=config.AUG_PROB_JPEG),
        A.ColorJitter(p=config.AUG_PROB_COLOR)
    ])

def get_val_transform(config): return None

def create_dataloaders(config, rank=0, world_size=1):
    train_ds = ForgeryDataset(config.TRAIN_AUTHENTIC_DIR, config.TRAIN_FORGED_DIR, config.TRAIN_MASK_DIR, get_train_transform(config), config)
    val_ds = ForgeryDataset(config.TEST_AUTHENTIC_DIR, config.TEST_FORGED_DIR, config.TEST_MASK_DIR, get_val_transform(config), config)
    
    train_loader = DataLoader(train_ds, batch_size=config.BATCH_SIZE, sampler=DistributedSampler(train_ds, num_replicas=world_size, rank=rank, shuffle=True), num_workers=4, pin_memory=True, drop_last=True)
    val_loader = DataLoader(val_ds, batch_size=config.BATCH_SIZE, sampler=DistributedSampler(val_ds, num_replicas=world_size, rank=rank, shuffle=False), num_workers=4, pin_memory=True)
    
    return train_loader, val_loader, DistributedSampler(train_ds, num_replicas=world_size, rank=rank), DistributedSampler(val_ds, num_replicas=world_size, rank=rank)

Overwriting dataset.py


In [177]:
%%writefile model.py
"""Hybrid-Stem Swin+UNet Model"""
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm

class ConvolutionalStem(nn.Module):
    """
    Takes 896x896 RGB input -> Outputs 224x224 rich features (128 channels)
    Preserves texture details before feeding to Swin.
    """
    def __init__(self, in_channels=3, base_width=64):
        super().__init__()
        
        # Block 1: 896 -> 448
        self.conv1 = nn.Sequential(
            nn.Conv2d(in_channels, base_width, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(base_width),
            nn.ReLU(inplace=True),
            nn.Conv2d(base_width, base_width, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(base_width),
            nn.ReLU(inplace=True)
        )
        
        # Block 2: 448 -> 224
        self.conv2 = nn.Sequential(
            nn.Conv2d(base_width, base_width*2, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(base_width*2),
            nn.ReLU(inplace=True),
            nn.Conv2d(base_width*2, base_width*2, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(base_width*2),
            nn.ReLU(inplace=True)
        )
        
    def forward(self, x):
        # x: (B, 3, 896, 896)
        x_448 = self.conv1(x)  # (B, 64, 448, 448)
        x_224 = self.conv2(x_448) # (B, 128, 224, 224)
        return x_224

class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.double_conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )
    def forward(self, x): return self.double_conv(x)

class Up(nn.Module):
    def __init__(self, in_channels, out_channels, skip_channels):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_channels, in_channels // 2, kernel_size=2, stride=2)
        self.conv = DoubleConv(in_channels // 2 + skip_channels, out_channels)
    
    def forward(self, x, skip=None):
        x = self.up(x)
        if skip is not None:
            if x.shape != skip.shape:
                x = F.interpolate(x, size=skip.shape[2:], mode='bilinear', align_corners=False)
            x = torch.cat([x, skip], dim=1)
        return self.conv(x)

class ContrastiveEmbeddingHead(nn.Module):
    def __init__(self, in_channels, embedding_dim):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, in_channels // 2, kernel_size=3, padding=1),
            nn.BatchNorm2d(in_channels // 2),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_channels // 2, embedding_dim, kernel_size=1)
        )
    def forward(self, x):
        return F.normalize(self.conv(x), p=2, dim=1)

class ActiveChannelHead(nn.Module):
    """Updated to match current model: Conv Neck -> Max/Avg Pool -> Linear"""
    def __init__(self, in_channels, num_channels=6):
        super().__init__()
        self.neck = nn.Sequential(
            nn.Conv2d(in_channels, 128, kernel_size=3, stride=2, padding=1, bias=False), # 224 -> 112
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 256, kernel_size=3, stride=2, padding=1, bias=False), # 112 -> 56
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True)
        )
        self.global_max_pool = nn.AdaptiveMaxPool2d(1)
        self.global_avg_pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(512, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),
            nn.Linear(256, num_channels)
        )
    
    def forward(self, x):
        x = self.neck(x)
        max_pool = self.global_max_pool(x).view(x.size(0), -1)
        avg_pool = self.global_avg_pool(x).view(x.size(0), -1)
        pooled = torch.cat([max_pool, avg_pool], dim=1)
        return self.fc(pooled)

class ForgeryDetectionModel(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        
        # 1. Stem
        self.stem = ConvolutionalStem(in_channels=3, base_width=64)
        
        # 2. Swin Encoder (Modified input channels)
        # We tell Swin to expect 128 channels (from Stem), not 3.
        self.encoder = timm.create_model(
            config.SWIN_MODEL,
            pretrained=True,
            features_only=True,
            in_chans=128,
            out_indices=[0, 1, 2, 3]
        )
        
        encoder_dims = [96, 192, 384, 768]
        decoder_dims = config.DECODER_CHANNELS
        
        # Decoder
        self.bottleneck = DoubleConv(encoder_dims[3], decoder_dims[0])
        self.up1 = Up(decoder_dims[0], decoder_dims[1], encoder_dims[2])
        self.up2 = Up(decoder_dims[1], decoder_dims[2], encoder_dims[1])
        self.up3 = Up(decoder_dims[2], decoder_dims[3], encoder_dims[0]) # 1/4 point (56x56)
        
        # Coordinate fusion at 1/4 point
        self.coord_fusion = DoubleConv(decoder_dims[3] + 2, decoder_dims[3])
        
        self.up4 = Up(decoder_dims[3], decoder_dims[4], 0)
        self.final_up = nn.Sequential(
            nn.ConvTranspose2d(decoder_dims[4], decoder_dims[4], kernel_size=2, stride=2),
            nn.BatchNorm2d(decoder_dims[4]),
            nn.ReLU(inplace=True)
        )
        
        # Heads
        # Contrastive Head placed at up3 (56x56) - Standard place
        self.contrastive_head = ContrastiveEmbeddingHead(decoder_dims[3], config.EMBEDDING_DIM)
        
        self.active_channel_head = ActiveChannelHead(decoder_dims[4], config.NUM_INSTANCES)
        
        self.segmentation_head = nn.Sequential(
            nn.Conv2d(decoder_dims[4], decoder_dims[4], kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(decoder_dims[4]),
            nn.ReLU(inplace=True),
            nn.Conv2d(decoder_dims[4], config.NUM_INSTANCES, kernel_size=1)
        )
        
        self.freeze_encoder()
        
    def freeze_encoder(self):
        for param in self.encoder.parameters(): param.requires_grad = False
        print("🧊 Encoder frozen (Stem is active)")
        
    def unfreeze_encoder(self):
        for param in self.encoder.parameters(): param.requires_grad = True
        print("🔥 Encoder unfrozen")

    def create_coordinate_channels(self, h, w, device):
        y_coords = torch.linspace(-1, 1, h, device=device).view(h, 1).expand(h, w)
        x_coords = torch.linspace(-1, 1, w, device=device).view(1, w).expand(h, w)
        return x_coords, y_coords

    def forward(self, x):
        # x: (B, 3, 896, 896)
        
        # Stem
        x_stem = self.stem(x) # (B, 128, 224, 224)
        
        # Encoder
        enc_features = self.encoder(x_stem)
        processed_features = []
        for feat in enc_features:
            if feat.dim() == 4 and feat.shape[1] < feat.shape[3]:
                feat = feat.permute(0, 3, 1, 2)
            processed_features.append(feat)
        enc_features = processed_features
        
        # Decoder
        x = self.bottleneck(enc_features[3])
        x = self.up1(x, enc_features[2])
        x = self.up2(x, enc_features[1])
        x = self.up3(x, enc_features[0]) # 56x56
        
        # Contrastive
        embeddings = self.contrastive_head(x)
        
        # Coord Fusion
        h, w = x.shape[2], x.shape[3]
        xc, yc = self.create_coordinate_channels(h, w, x.device)
        xc = xc.unsqueeze(0).expand(x.shape[0], -1, -1).unsqueeze(1)
        yc = yc.unsqueeze(0).expand(x.shape[0], -1, -1).unsqueeze(1)
        x = torch.cat([x, xc, yc], dim=1)
        x = self.coord_fusion(x)
        
        x = self.up4(x)
        x = self.final_up(x) # 224x224
        
        # Heads
        active_logits = self.active_channel_head(x)
        seg_logits = self.segmentation_head(x)
        segmentation = torch.sigmoid(seg_logits)
        
        return {
            'segmentation': segmentation,
            'segmentation_logits': seg_logits,
            'embeddings': embeddings,
            'active_logits': active_logits
        }

def create_model(config): return ForgeryDetectionModel(config)

Overwriting model.py


In [178]:
%%writefile losses.py
"""Losses with Memory Leak Fix"""
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy.optimize import linear_sum_assignment
import numpy as np

class DiceLoss(nn.Module):
    def __init__(self, smooth=1.0):
        super().__init__()
        self.smooth = smooth
    def forward(self, pred, target):
        pred = pred.contiguous().view(pred.size(0), -1)
        target = target.contiguous().view(target.size(0), -1)
        intersection = (pred * target).sum(dim=1)
        dice = (2. * intersection + self.smooth) / (pred.sum(dim=1) + target.sum(dim=1) + self.smooth)
        return 1 - dice.mean()

class SegmentationFocalLoss(nn.Module):
    def __init__(self, alpha=0.99, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
    def forward(self, pred_logits, target):
        pred_logits = pred_logits.contiguous().view(-1)
        target = target.contiguous().view(-1)
        bce = F.binary_cross_entropy_with_logits(pred_logits, target, reduction='none')
        pred_probs = torch.sigmoid(pred_logits)
        pt = torch.where(target == 1, pred_probs, 1 - pred_probs)
        loss = torch.where(target == 1, self.alpha, 1 - self.alpha) * (1 - pt) ** self.gamma * bce
        return loss.mean()

class ContrastiveLoss(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.max_samples_bg = config.CONTRASTIVE_SAMPLES_BACKGROUND
        self.max_samples_inst = config.CONTRASTIVE_SAMPLES_PER_INSTANCE
        self.margin_bg_fg = config.CONTRASTIVE_MARGIN_BG_FG
        self.margin_fg_fg = config.CONTRASTIVE_MARGIN_FG_FG
        self.w_bg_fg = config.CONTRAST_WEIGHT_BG_FG
        self.w_fg_fg = config.CONTRAST_WEIGHT_FG_FG

    def forward(self, embeddings, masks):
        B, D, H, W = embeddings.shape
        device = embeddings.device
        masks_down = F.max_pool2d(masks, kernel_size=4, stride=4)
        total_loss = 0.0
        valid_batches = 0
        for b in range(B):
            emb = embeddings[b].view(D, -1).t()
            curr_masks = masks_down[b]
            label_map = torch.zeros((H, W), device=device, dtype=torch.long)
            active_ids = []
            for i in range(curr_masks.shape[0]):
                mask_i = curr_masks[i]
                if mask_i.sum() > 0:
                    label_map[mask_i > 0.5] = (i + 1)
                    active_ids.append(i + 1)
            label_flat = label_map.flatten()
            sampled_embeddings, sampled_labels = [], []
            bg_indices = torch.where(label_flat == 0)[0]
            if len(bg_indices) > 0:
                n_bg = min(len(bg_indices), self.max_samples_bg)
                perm = torch.randperm(len(bg_indices), device=device)[:n_bg]
                sampled_embeddings.append(emb[bg_indices[perm]])
                sampled_labels.append(label_flat[bg_indices[perm]])
            for inst_id in active_ids:
                inst_indices = torch.where(label_flat == inst_id)[0]
                if len(inst_indices) > 0:
                    n_inst = min(len(inst_indices), self.max_samples_inst)
                    perm = torch.randperm(len(inst_indices), device=device)[:n_inst]
                    sampled_embeddings.append(emb[inst_indices[perm]])
                    sampled_labels.append(label_flat[inst_indices[perm]])
            if len(sampled_labels) == 0: continue
            anchors = torch.cat(sampled_embeddings, dim=0)
            anchor_labels = torch.cat(sampled_labels, dim=0)
            if anchors.shape[0] < 2: continue
            dist_matrix = torch.cdist(anchors, anchors, p=2)
            mask_same_class = anchor_labels.unsqueeze(0) == anchor_labels.unsqueeze(1)
            is_bg = anchor_labels == 0
            mask_bg_fg = is_bg.unsqueeze(0) ^ is_bg.unsqueeze(1)
            mask_fg_fg = (~mask_same_class) & (~mask_bg_fg)
            mask_same_class.fill_diagonal_(False)
            pos_loss = dist_matrix[mask_same_class].mean() if mask_same_class.sum() > 0 else 0.0
            neg_loss_bg = F.relu(self.margin_bg_fg - dist_matrix[mask_bg_fg]).mean() if mask_bg_fg.sum() > 0 else 0.0
            neg_loss_fg = F.relu(self.margin_fg_fg - dist_matrix[mask_fg_fg]).mean() if mask_fg_fg.sum() > 0 else 0.0
            total_loss += pos_loss + (self.w_bg_fg * neg_loss_bg) + (self.w_fg_fg * neg_loss_fg)
            valid_batches += 1
        return total_loss / valid_batches if valid_batches > 0 else 0.0 * embeddings.sum()

class FocalLoss(nn.Module):
    def __init__(self, alpha=0.75, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
    def forward(self, logits, targets):
        bce_loss = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')
        probs = torch.sigmoid(logits)
        pt = torch.where(targets == 1, probs, 1 - probs)
        loss = torch.where(targets == 1, self.alpha, 1 - self.alpha) * (1 - pt) ** self.gamma * bce_loss
        return loss.mean()

def hungarian_matching(pred_masks, gt_masks):
    B = pred_masks.shape[0]
    indices = []
    for b in range(B):
        pred, gt = pred_masks[b], gt_masks[b]
        active_gt = [i for i in range(6) if gt[i].sum() > 0]
        cost_matrix = np.zeros((6, 6))
        for i in range(6):
            p_flat = pred[i].flatten().detach().cpu().numpy()
            for j in range(len(active_gt)):
                g_flat = gt[active_gt[j]].flatten().detach().cpu().numpy()
                intersection = (p_flat * g_flat).sum()
                union = p_flat.sum() + g_flat.sum()
                dice = (2.0 * intersection) / (union + 1e-6) if union > 0 else 0.0
                cost_matrix[i, j] = 1.0 - dice
            for j in range(len(active_gt), 6):
                cost_matrix[i, j] = p_flat.sum() / (p_flat.shape[0] + 1e-6)
        row, col = linear_sum_assignment(cost_matrix)
        mapping = {r: (active_gt[c] if c < len(active_gt) else -1) for r, c in zip(row, col)}
        indices.append(mapping)
    return indices

class CombinedLoss(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.dice_loss = DiceLoss()
        self.seg_focal_loss = SegmentationFocalLoss(config.SEG_FOCAL_ALPHA, config.SEG_FOCAL_GAMMA)
        self.contrastive_loss = ContrastiveLoss(config)
        self.focal_loss = FocalLoss(config.FOCAL_ALPHA, config.FOCAL_GAMMA)
        
    def forward(self, outputs, targets):
        pred_masks = outputs['segmentation']
        seg_logits = outputs['segmentation_logits']
        active_logits = outputs['active_logits']
        gt_masks = targets['masks'] # Already 224x224 from Dataset
        
        with torch.no_grad(): mappings = hungarian_matching(pred_masks, gt_masks)
        
        dice_acc, seg_acc, matches = 0.0, 0.0, 0
        active_targets = torch.zeros_like(active_logits)
        
        for b in range(pred_masks.shape[0]):
            mapping = mappings[b]
            for pid in range(6):
                gid = mapping[pid]
                if gid != -1:
                    active_targets[b, pid] = 1.0
                    dice_acc += self.dice_loss(pred_masks[b, pid].unsqueeze(0), gt_masks[b, gid].unsqueeze(0))
                    seg_acc += self.seg_focal_loss(seg_logits[b, pid], gt_masks[b, gid])
                    matches += 1
        
        dice_loss = dice_acc / matches if matches > 0 else 0.0
        seg_loss = seg_acc / matches if matches > 0 else 0.0
        focal_loss = self.focal_loss(active_logits, active_targets)
        cont_loss = self.contrastive_loss(outputs['embeddings'], gt_masks)
        
        total = (self.config.LOSS_WEIGHT_DICE * dice_loss + 
                 self.config.LOSS_WEIGHT_SEGMENTATION_FOCAL * seg_loss + 
                 self.config.LOSS_WEIGHT_ACTIVE * focal_loss + 
                 self.config.LOSS_WEIGHT_CONTRASTIVE * cont_loss)
        
        # FIX: DETACH TENSORS FROM GRAPH FOR LOGGING
        return total, {
            'dice': dice_loss.item() if torch.is_tensor(dice_loss) else dice_loss,
            'seg_focal': seg_loss.item() if torch.is_tensor(seg_loss) else seg_loss,
            'focal': focal_loss.item() if torch.is_tensor(focal_loss) else focal_loss,
            'contrastive': cont_loss.item() if torch.is_tensor(cont_loss) else cont_loss
        }

Overwriting losses.py


In [179]:
%%writefile train.py
"""Training script: Hybrid Stem + Streaming Validation"""
import os
import torch
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.cuda.amp import autocast, GradScaler
from tqdm import tqdm
import numpy as np
import random
import json
from pathlib import Path

from config import Config
from model import create_model
from losses import CombinedLoss
from dataset import create_dataloaders
from utils import oF1_score

def setup_ddp(rank, world_size):
    os.environ['MASTER_ADDR'] = 'localhost'
    os.environ['MASTER_PORT'] = '12355'
    dist.init_process_group(Config.BACKEND, rank=rank, world_size=world_size)
    torch.cuda.set_device(rank)

def cleanup_ddp(): dist.destroy_process_group()

def save_checkpoint(model, optimizer, scheduler, scaler, epoch, best_of1, checkpoint_dir, is_best=False):
    Path(checkpoint_dir).mkdir(parents=True, exist_ok=True)
    checkpoint = {
        'epoch': epoch,
        'model_state_dict': model.module.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'scaler_state_dict': scaler.state_dict(),
        'best_of1': best_of1
    }
    torch.save(checkpoint, os.path.join(checkpoint_dir, 'current_checkpoint.pth'))
    if is_best: torch.save(checkpoint, os.path.join(checkpoint_dir, 'best_checkpoint.pth'))

def compute_of1_score(model, dataloader, config, device, rank):
    """Streaming Validation (Memory Safe)"""
    model.eval()
    authentic_scores, forged_scores = [], []
    
    with torch.no_grad():
        pbar = tqdm(dataloader, desc="Computing OF1", disable=(rank != 0))
        for batch in pbar:
            if random.random() > config.VAL_SAMPLE_RATIO: continue
            
            inputs = batch['input'].to(device)
            gt_masks = batch['masks'].to(device)
            is_forged = batch['is_forged']
            
            outputs = model(inputs)
            pred_masks = outputs['segmentation']
            active_probs = torch.sigmoid(outputs['active_logits'])
            active_channels = (active_probs > config.ACTIVE_CHANNEL_THRESHOLD).cpu().numpy()
            
            for b in range(inputs.shape[0]):
                active_idx = np.where(active_channels[b])[0]
                preds = [(pred_masks[b, i].cpu().numpy() > config.MASK_THRESHOLD).astype(np.float32) for i in active_idx]
                gts = [gt_masks[b, i].cpu().numpy() for i in range(6) if gt_masks[b, i].sum() > 0]
                
                if not gts: score = 1.0 if not preds else 0.0
                else: score = 0.0 if not preds else oF1_score(preds, gts)
                
                if is_forged[b].item() > 0.5: forged_scores.append(score)
                else: authentic_scores.append(score)
    
    model.train()
    overall = np.mean(authentic_scores + forged_scores) if (authentic_scores + forged_scores) else 0.0
    return overall, np.mean(authentic_scores) if authentic_scores else 0.0, np.mean(forged_scores) if forged_scores else 0.0

def train_one_epoch(model, loader, criterion, optimizer, scheduler, scaler, epoch, config, device, rank):
    model.train()
    total_loss = 0.0
    loss_sum = {'dice': 0.0, 'seg_focal': 0.0, 'contrastive': 0.0, 'focal': 0.0}
    
    pbar = tqdm(loader, desc=f"Epoch {epoch}", disable=(rank != 0))
    for i, batch in enumerate(pbar):
        inputs = batch['input'].to(device)
        targets = {'masks': batch['masks'].to(device)}
        
        with autocast():
            outputs = model(inputs)
            loss, loss_dict = criterion(outputs, targets)
            loss = loss / config.GRADIENT_ACCUMULATION_STEPS
            
        scaler.scale(loss).backward()
        
        if (i + 1) % config.GRADIENT_ACCUMULATION_STEPS == 0:
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            
        total_loss += loss.item() * config.GRADIENT_ACCUMULATION_STEPS
        for k in loss_dict: loss_sum[k] += loss_dict[k]
        
        if rank == 0:
            pbar.set_postfix({'loss': f"{loss.item()*config.GRADIENT_ACCUMULATION_STEPS:.4f}", **{k: f"{v:.3f}" for k, v in loss_dict.items()}})
            
    return total_loss / len(loader), {k: v / len(loader) for k, v in loss_sum.items()}

def train_ddp(rank, world_size, config, smoke_test=False):
    setup_ddp(rank, world_size)
    if smoke_test: config.set_smoke_test(True)
    if rank == 0: config.print_config()
    
    train_loader, val_loader, train_sampler, _ = create_dataloaders(config, rank, world_size)
    model = DDP(create_model(config).to(rank), device_ids=[rank], find_unused_parameters=True)
    criterion = CombinedLoss(config).to(rank)
    optimizer = AdamW(model.parameters(), lr=config.LEARNING_RATE, weight_decay=config.WEIGHT_DECAY)
    scheduler = CosineAnnealingLR(optimizer, T_max=config.T_MAX, eta_min=config.ETA_MIN)
    scaler = GradScaler()
    
    best_of1 = -1.0
    for epoch in range(1, config.NUM_EPOCHS + 1):
        train_sampler.set_epoch(epoch)
        if epoch == config.ENCODER_FREEZE_EPOCHS + 1: model.module.unfreeze_encoder()
        
        loss, loss_dict = train_one_epoch(model, train_loader, criterion, optimizer, scheduler, scaler, epoch, config, rank, rank)
        scheduler.step()
        
        if (epoch % config.VAL_FREQUENCY == 0 or epoch == 1 or epoch == config.NUM_EPOCHS):
            overall, auth, forg = compute_of1_score(model, val_loader, config, rank, rank)
            if rank == 0:
                print(f"Epoch {epoch} | Loss: {loss:.4f} | OF1: {overall:.4f} (Auth: {auth:.4f}, Forg: {forg:.4f})")
                if overall > best_of1:
                    best_of1 = overall
                    save_checkpoint(model, optimizer, scheduler, scaler, epoch, best_of1, config.CHECKPOINT_DIR, is_best=True)
    cleanup_ddp()

def main():
    import sys, torch.multiprocessing as mp
    mp.spawn(train_ddp, args=(Config.WORLD_SIZE, Config, '--smoke-test' in sys.argv), nprocs=Config.WORLD_SIZE, join=True)

if __name__ == '__main__': main()

Overwriting train.py


In [180]:
%%writefile test.py
"""Detailed Testing Script - Robust to Missing History"""
import os
import torch
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import DataLoader
from torch.utils.data.distributed import DistributedSampler
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from tqdm import tqdm
import cv2
from sklearn.metrics import roc_auc_score, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
import json
import warnings

from config import Config
from model import create_model
from dataset import ForgeryDataset, get_val_transform
from utils import oF1_score

warnings.filterwarnings("ignore")

def setup_ddp(rank, world_size):
    os.environ['MASTER_ADDR'] = 'localhost'
    os.environ['MASTER_PORT'] = '12356'
    dist.init_process_group(Config.BACKEND, rank=rank, world_size=world_size)
    torch.cuda.set_device(rank)

def cleanup_ddp(): dist.destroy_process_group()

def load_thresholds(config_dir):
    path = os.path.join(config_dir, 'best_thresholds.json')
    if os.path.exists(path):
        with open(path, 'r') as f:
            data = json.load(f)
        return data['active_threshold'], data['mask_threshold']
    else:
        return 0.5, 0.5

def load_best_model(config, device):
    best_path = os.path.join(config.CHECKPOINT_DIR, 'best_checkpoint.pth')
    curr_path = os.path.join(config.CHECKPOINT_DIR, 'current_checkpoint.pth')
    path = best_path if os.path.exists(best_path) else curr_path
    
    if not os.path.exists(path): 
        if device == 0: print("⚠️ No checkpoint found, initializing random model")
        return create_model(config).to(device)
    
    if device == 0: print(f"✅ Loading checkpoint: {path}")
    model = create_model(config).to(device)
    ckpt = torch.load(path, map_location=f'cuda:{device}', weights_only=False)
    model.load_state_dict(ckpt['model_state_dict'])
    model.eval()
    return model

def plot_training_history(output_dir):
    """Attempts to plot history if file exists"""
    history_path = os.path.join(output_dir, 'training_history.json')
    if not os.path.exists(history_path):
        print(f"⚠️  Training history not found (likely lost during crash). Skipping curves.")
        return

    print(f"📈 Plotting training history...")
    with open(history_path, 'r') as f:
        history = json.load(f)
        
    epochs = history.get('epochs', [])
    if not epochs: return

    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    
    # 1. Losses
    ax = axes[0, 0]
    if 'train_loss_total' in history:
        ax.plot(epochs, history['train_loss_total'], label='Total', color='black', linewidth=2)
    for k in ['train_loss_dice', 'train_loss_seg_focal', 'train_loss_contrastive', 'train_loss_focal']:
        if k in history:
            ax.plot(epochs, history[k], label=k.replace('train_loss_', ''), linestyle='--')
    ax.set_title('Training Losses')
    ax.set_xlabel('Epoch')
    ax.legend()
    ax.grid(True, alpha=0.3)

    # 2. Validation OF1
    ax = axes[0, 1]
    if 'val_of1_overall' in history:
        ax.plot(epochs, history['val_of1_overall'], label='Overall', color='purple', linewidth=2, marker='o')
    if 'val_of1_authentic' in history:
        ax.plot(epochs, history['val_of1_authentic'], label='Authentic', color='green', linestyle='--')
    if 'val_of1_forged' in history:
        ax.plot(epochs, history['val_of1_forged'], label='Forged', color='red', linestyle='--')
    ax.set_title('Validation OF1 Scores')
    ax.set_xlabel('Epoch')
    ax.legend()
    ax.grid(True, alpha=0.3)

    # 3. Learning Rate
    ax = axes[1, 0]
    if 'lr' in history:
        ax.plot(epochs, history['lr'], color='orange')
    ax.set_title('Learning Rate')
    ax.set_xlabel('Epoch')
    ax.set_yscale('log')
    ax.grid(True, alpha=0.3)

    ax = axes[1, 1]; ax.axis('off')
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, 'training_curves.png'))
    plt.close()

def evaluate_ddp(rank, world_size, dataset, config, thresholds, dataset_name, save_worst=True):
    active_thresh, mask_thresh = thresholds
    # CRITICAL FIX: Create directory here to prevent race conditions
    if rank == 0:
        Path(config.OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
    
    sampler = DistributedSampler(dataset, num_replicas=world_size, rank=rank, shuffle=False)
    loader = DataLoader(dataset, batch_size=config.BATCH_SIZE, sampler=sampler, num_workers=4)
    model = load_best_model(config, rank)
    model = DDP(model, device_ids=[rank])
    model.eval()
    
    local_metrics = []
    local_candidates = []
    
    with torch.no_grad():
        pbar = tqdm(loader, desc=f"Eval {dataset_name}", disable=(rank!=0))
        for batch in pbar:
            inputs = batch['input'].to(rank)
            gt_masks = batch['masks'].to(rank)
            outputs = model(inputs)
            
            active_ch = (torch.sigmoid(outputs['active_logits']) > active_thresh).cpu().numpy()
            pred_masks = outputs['segmentation']
            
            B = inputs.shape[0]
            for b in range(B):
                preds = []
                for i in np.where(active_ch[b])[0]:
                    m = pred_masks[b, i].cpu().numpy()
                    preds.append((m > mask_thresh).astype(np.float32))
                
                gts = []
                for i in range(config.NUM_INSTANCES):
                    m = gt_masks[b, i].cpu().numpy()
                    if m.sum() > 0: gts.append(m)
                
                if not gts:
                    score = 1.0 if not preds else 0.0
                else:
                    score = 0.0 if not preds else oF1_score(preds, gts)
                
                if np.isnan(score): score = 0.0
                
                is_forged = int(batch['is_forged'][b].item())
                
                local_metrics.append({
                    'is_forged': is_forged,
                    'pred_forged': int(len(preds) > 0),
                    'of1': float(score)
                })
                
                if save_worst and is_forged:
                    local_candidates.append({
                        'of1': float(score),
                        'preds': preds, 'gts': gts,
                        'img_thumb': cv2.resize(inputs[b].cpu().permute(1,2,0).numpy(), (224,224))
                    })

    gathered = [None]*world_size
    dist.all_gather_object(gathered, local_metrics)
    
    final_metrics = None
    worst_final = None
    
    if rank == 0:
        all_data = [x for sub in gathered for x in sub]
        
        auth = [r for r in all_data if r['is_forged'] == 0]
        forg = [r for r in all_data if r['is_forged'] == 1]
        
        of1_all = np.mean([r['of1'] for r in all_data]) if all_data else 0.0
        of1_auth = np.mean([r['of1'] for r in auth]) if auth else 0.0
        of1_forg = np.mean([r['of1'] for r in forg]) if forg else 0.0
        
        y_true = [r['is_forged'] for r in all_data]
        y_pred = [r['pred_forged'] for r in all_data]
        
        cm = confusion_matrix(y_true, y_pred)
        tn, fp, fn, tp = cm.ravel() if cm.size == 4 else (0,0,0,0)
        
        final_metrics = {
            'segmentation': {'overall_of1': of1_all, 'authentic_of1': of1_auth, 'forged_of1': of1_forg},
            'detection': {
                'accuracy': accuracy_score(y_true, y_pred),
                'precision': precision_score(y_true, y_pred, zero_division=0),
                'recall': recall_score(y_true, y_pred, zero_division=0),
                'f1': f1_score(y_true, y_pred, zero_division=0),
                'auc': roc_auc_score(y_true, y_pred) if len(set(y_true))>1 else 0.0,
                'confusion_matrix': {'tn': int(tn), 'fp': int(fp), 'fn': int(fn), 'tp': int(tp)}
            }
        }
        
        # Save Confusion Matrix
        plt.figure(figsize=(5,4))
        sns.heatmap([[tn, fp],[fn, tp]], annot=True, fmt='d', cmap='Blues', 
                   xticklabels=['Auth', 'Forg'], yticklabels=['Auth', 'Forg'])
        plt.title(f'{dataset_name} Confusion Matrix')
        plt.ylabel('Actual')
        plt.xlabel('Predicted')
        plt.savefig(os.path.join(config.OUTPUT_DIR, f'cm_{dataset_name.split()[0]}.png'))
        plt.close()

    if save_worst:
        local_candidates.sort(key=lambda x: x['of1'])
        local_bw = local_candidates[:config.SAVE_TOP_WORST]
        gather_w = [None]*world_size
        dist.all_gather_object(gather_w, local_bw)
        
        if rank == 0:
            all_w = [x for sub in gather_w for x in sub]
            all_w.sort(key=lambda x: x['of1'])
            worst_final = all_w[:config.SAVE_TOP_WORST]
            
    return final_metrics, worst_final

def vis_worst(worst, output_dir):
    vis_dir = os.path.join(output_dir, 'worst_predictions')
    Path(vis_dir).mkdir(parents=True, exist_ok=True)
    colors = [(1,0,0), (0,1,0), (0,0,1), (1,1,0)]
    print(f"🖼️ Saving {len(worst)} worst cases...")
    for i, res in enumerate(worst):
        fig, ax = plt.subplots(1,3,figsize=(12,4))
        ax[0].imshow(res['img_thumb']); ax[0].axis('off')
        
        gtv = res['img_thumb'].copy()
        for j, m in enumerate(res['gts']):
            c = np.zeros_like(gtv); c[:,:,j%3] = m
            gtv = cv2.addWeighted(gtv, 1, c, 0.5, 0)
        ax[1].imshow(gtv); ax[1].set_title('GT'); ax[1].axis('off')
        
        prv = res['img_thumb'].copy()
        for j, m in enumerate(res['preds']):
            c = np.zeros_like(prv); c[:,:,j%3] = m
            prv = cv2.addWeighted(prv, 1, c, 0.5, 0)
        ax[2].imshow(prv); ax[2].set_title(f"Pred {res['of1']:.2f}"); ax[2].axis('off')
        plt.savefig(os.path.join(vis_dir, f"worst_{i}.png")); plt.close()

def run_test(rank, world_size):
    setup_ddp(rank, world_size)
    
    # 1. Create Directory (Rank 0)
    if rank == 0:
        Path(Config.OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
        # Try to plot history if it exists
        plot_training_history(Config.OUTPUT_DIR)
    dist.barrier()
    
    # 2. Load Thresholds
    thr = load_thresholds(Config.OUTPUT_DIR) if rank==0 else (0.5,0.5)
    thr = [thr]; dist.broadcast_object_list(thr, src=0); thr=thr[0]
    
    test_ds = ForgeryDataset(Config.TEST_AUTHENTIC_DIR, Config.TEST_FORGED_DIR, Config.TEST_MASK_DIR, get_val_transform(Config), Config)
    hq_ds = ForgeryDataset(Config.HQ_AUTHENTIC_DIR, Config.HQ_FORGED_DIR, Config.HQ_MASK_DIR, get_val_transform(Config), Config)
    
    tm, tw = evaluate_ddp(rank, world_size, test_ds, Config, thr, "Test Set", True)
    hm, _ = evaluate_ddp(rank, world_size, hq_ds, Config, thr, "HQ Set", False)
    
    if rank == 0:
        print("\n" + "="*50 + "\nFINAL REPORT\n" + "="*50)
        for n, m in [("TEST SET", tm), ("HQ SET", hm)]:
            s, d, c = m['segmentation'], m['detection'], m['detection']['confusion_matrix']
            print(f"\n🔹 {n}")
            print(f"   Seg OF1:  All:{s['overall_of1']:.4f} | Auth:{s['authentic_of1']:.4f} | Forg:{s['forged_of1']:.4f}")
            print(f"   Detect:   Acc:{d['accuracy']:.4f} | F1:{d['f1']:.4f} | AUC:{d['auc']:.4f}")
            print(f"   Matrix:   TN:{c['tn']} FP:{c['fp']} | FN:{c['fn']} TP:{c['tp']}")
            
        with open(os.path.join(Config.OUTPUT_DIR, 'final_metrics.json'), 'w') as f:
            json.dump({'test': tm, 'hq': hm}, f, indent=2)
        if tw: vis_worst(tw, Config.OUTPUT_DIR)
    cleanup_ddp()

if __name__ == '__main__':
    import torch.multiprocessing as mp
    mp.spawn(run_test, args=(Config.WORLD_SIZE,), nprocs=Config.WORLD_SIZE, join=True)

Overwriting test.py


In [181]:
%%writefile threshold.py
"""
Threshold Tuning on Sampled Training Data (DDP)
Target: 15% of Train Data with 30/70 Auth/Forged Split
"""
import os
import torch
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import DataLoader, Subset
from torch.utils.data.distributed import DistributedSampler
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import json
import random
from pathlib import Path

from config import Config
from model import create_model
from dataset import ForgeryDataset, get_val_transform
from utils import oF1_score

def setup_ddp(rank, world_size):
    os.environ['MASTER_ADDR'] = 'localhost'
    os.environ['MASTER_PORT'] = '12357'
    dist.init_process_group(Config.BACKEND, rank=rank, world_size=world_size)
    torch.cuda.set_device(rank)

def cleanup_ddp(): dist.destroy_process_group()

def load_model(config, device):
    path = os.path.join(config.CHECKPOINT_DIR, 'best_checkpoint.pth')
    if not os.path.exists(path): path = os.path.join(config.CHECKPOINT_DIR, 'current_checkpoint.pth')
    
    # weights_only=False for numpy scalars
    ckpt = torch.load(path, map_location=f'cuda:{device}', weights_only=False)
    
    model = create_model(config).to(device)
    model.load_state_dict(ckpt['model_state_dict'])
    model.eval()
    return model

def sample_dataset_indices(dataset, config):
    """
    Custom Sampling Logic:
    1. Filter indices by class (Authentic vs Forged)
    2. Calculate target counts based on Ratio (15%) and Distribution (30/70)
    """
    total_samples = len(dataset)
    target_size = int(total_samples * config.TUNING_SAMPLE_RATIO)
    
    n_auth_target = int(target_size * config.TUNING_AUTHENTIC_RATIO)
    n_forg_target = target_size - n_auth_target
    
    # Separate indices
    auth_indices = [i for i, s in enumerate(dataset.samples) if not s['is_forged']]
    forg_indices = [i for i, s in enumerate(dataset.samples) if s['is_forged']]
    
    # Cap at available
    n_auth_target = min(n_auth_target, len(auth_indices))
    n_forg_target = min(n_forg_target, len(forg_indices))
    
    # Sample
    random.seed(42) # Deterministic
    sel_auth = random.sample(auth_indices, n_auth_target)
    sel_forg = random.sample(forg_indices, n_forg_target)
    
    return sel_auth + sel_forg

def threshold_search_ddp(rank, world_size):
    setup_ddp(rank, world_size)
    
    # 1. Load TRAINING Dataset (As requested for tuning)
    # We use val_transform because we want deterministic evaluation, not augmentation
    ds = ForgeryDataset(
        Config.TRAIN_AUTHENTIC_DIR, 
        Config.TRAIN_FORGED_DIR, 
        Config.TRAIN_MASK_DIR, 
        get_val_transform(Config), 
        Config
    )
    
    # 2. Sample Indices (Rank 0 decides, others follow)
    if rank == 0:
        subset_indices = sample_dataset_indices(ds, Config)
        print(f"📊 Tuning on {len(subset_indices)} samples ({Config.TUNING_SAMPLE_RATIO*100}%)")
        print(f"   Distribution: {Config.TUNING_AUTHENTIC_RATIO*100}/{Config.TUNING_FORGED_RATIO*100} Auth/Forged")
    else:
        subset_indices = []
    
    obj_list = [subset_indices]
    dist.broadcast_object_list(obj_list, src=0)
    subset_indices = obj_list[0]
    
    # Create Subset
    ds = Subset(ds, subset_indices)
    
    sampler = DistributedSampler(ds, num_replicas=world_size, rank=rank, shuffle=False)
    loader = DataLoader(ds, batch_size=Config.BATCH_SIZE, sampler=sampler, num_workers=4)
    
    model = load_model(Config, rank)
    model = DDP(model, device_ids=[rank])
    
    # Grid Search: Map (active_t, mask_t) -> [scores...]
    local_scores = { (at, mt): [] for at in Config.TEST_ACTIVE_THRESHOLDS for mt in Config.TEST_MASK_THRESHOLDS }
    
    with torch.no_grad():
        for batch in tqdm(loader, desc="Tuning", disable=(rank!=0)):
            inputs = batch['input'].to(rank)
            gt_masks = batch['masks'].to(rank)
            
            outputs = model(inputs)
            pred_masks = outputs['segmentation']     # (B, 6, 224, 224)
            active_probs = torch.sigmoid(outputs['active_logits'])
            
            B = inputs.shape[0]
            for b in range(B):
                gts = [gt_masks[b, i].cpu().numpy() for i in range(6) if gt_masks[b, i].sum() > 0]
                
                # Pre-fetch arrays
                p_masks_np = [pred_masks[b, i].cpu().numpy() for i in range(6)]
                act_probs_np = active_probs[b].cpu().numpy()
                
                for at in Config.TEST_ACTIVE_THRESHOLDS:
                    active_idx = np.where(act_probs_np > at)[0]
                    
                    for mt in Config.TEST_MASK_THRESHOLDS:
                        preds = []
                        for i in active_idx:
                            preds.append((p_masks_np[i] > mt).astype(np.float32))
                        
                        if not gts: score = 1.0 if not preds else 0.0
                        else: score = 0.0 if not preds else oF1_score(preds, gts)
                        
                        local_scores[(at, mt)].append(score)

    gathered = [None for _ in range(world_size)]
    dist.all_gather_object(gathered, local_scores)
    
    if rank == 0:
        final_scores = {k: [] for k in local_scores.keys()}
        for rank_dict in gathered:
            for k, v in rank_dict.items():
                final_scores[k].extend(v)
        
        results = {}
        best_cfg = None
        best_score = -1.0
        
        # Search for best mean
        matrix = np.zeros((len(Config.TEST_ACTIVE_THRESHOLDS), len(Config.TEST_MASK_THRESHOLDS)))
        
        for i, at in enumerate(Config.TEST_ACTIVE_THRESHOLDS):
            for j, mt in enumerate(Config.TEST_MASK_THRESHOLDS):
                scores = final_scores[(at, mt)]
                mean_score = np.mean(scores) if scores else 0.0
                
                results[f"{at}_{mt}"] = mean_score
                matrix[i, j] = mean_score
                
                if mean_score > best_score:
                    best_score = mean_score
                    best_cfg = (at, mt)
        
        print(f"\n🏆 BEST CONFIG FOUND: Active={best_cfg[0]}, Mask={best_cfg[1]} -> OF1: {best_score:.4f}")
        
        # Save Best Config for test.py to read
        Path(Config.OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
        
        best_params = {
            'active_threshold': best_cfg[0],
            'mask_threshold': best_cfg[1],
            'tuning_of1': best_score
        }
        
        with open(os.path.join(Config.OUTPUT_DIR, 'best_thresholds.json'), 'w') as f:
            json.dump(best_params, f, indent=2)
            
        # Plot Heatmap
        plt.figure(figsize=(10, 8))
        plt.imshow(matrix, cmap='viridis', aspect='auto')
        plt.colorbar(label='OF1 Score')
        plt.xticks(range(len(Config.TEST_MASK_THRESHOLDS)), Config.TEST_MASK_THRESHOLDS)
        plt.yticks(range(len(Config.TEST_ACTIVE_THRESHOLDS)), Config.TEST_ACTIVE_THRESHOLDS)
        plt.xlabel('Mask Threshold')
        plt.ylabel('Active Threshold')
        plt.title(f'Tuning Heatmap (Best: {best_score:.4f})')
        
        for i in range(len(Config.TEST_ACTIVE_THRESHOLDS)):
            for j in range(len(Config.TEST_MASK_THRESHOLDS)):
                val = matrix[i, j]
                color = 'black' if val > (matrix.max() * 0.7) else 'white'
                plt.text(j, i, f"{val:.3f}", ha='center', va='center', color=color)
                
        plt.savefig(os.path.join(Config.OUTPUT_DIR, 'tuning_heatmap.png'))
        print(f"✅ Tuning Complete. Best params saved to {Config.OUTPUT_DIR}/best_thresholds.json")
        
    cleanup_ddp()

if __name__ == '__main__':
    import torch.multiprocessing as mp
    mp.spawn(threshold_search_ddp, args=(Config.WORLD_SIZE,), nprocs=Config.WORLD_SIZE, join=True)

Overwriting threshold.py


In [182]:
# !python train.py --smoke-test

In [183]:
# !python train.py

In [184]:
!python threshold.py

In [185]:
!python test.py

/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

In [188]:
import json
import matplotlib.pyplot as plt
import os
from IPython.display import Image, display

def plot_history():
    history_path = '/kaggle/working/outputs/training_history.json'
    output_path = '/kaggle/working/outputs/training_curves.png'
    
    if not os.path.exists(history_path):
        print(f"❌ No history file found at {history_path}")
        return

    with open(history_path, 'r') as f:
        history = json.load(f)

    epochs = history.get('epochs', [])
    if not epochs:
        print("❌ History file is empty or corrupted.")
        return

    # Create 2x2 Plot
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    fig.suptitle('Training Metrics', fontsize=16)

    # 1. Losses (Detailed)
    if 'train_loss_dice' in history:
        axes[0, 0].plot(epochs, history['train_loss_dice'], label='Dice', marker='.')
        axes[0, 0].plot(epochs, history['train_loss_seg_focal'], label='Seg Focal', marker='.')
        axes[0, 0].plot(epochs, history['train_loss_contrastive'], label='Contrastive', marker='.')
        axes[0, 0].plot(epochs, history['train_loss_focal'], label='Active Focal', marker='.')
        axes[0, 0].set_title('Component Losses')
        axes[0, 0].set_xlabel('Epoch')
        axes[0, 0].set_ylabel('Loss')
        axes[0, 0].legend()
        axes[0, 0].grid(True, alpha=0.3)

    # 2. Total Loss
    if 'train_loss_total' in history:
        axes[0, 1].plot(epochs, history['train_loss_total'], color='black', linewidth=2)
        axes[0, 1].set_title('Total Loss')
        axes[0, 1].set_xlabel('Epoch')
        axes[0, 1].grid(True, alpha=0.3)

    # 3. Validation Metrics (OF1)
    if 'val_of1' in history:
        axes[1, 0].plot(epochs, history['val_of1'], color='purple', marker='o', label='Overall OF1')
        
        # Note: Previous train.py missed saving split metrics, checking if they exist just in case
        if 'val_auth' in history: 
            axes[1, 0].plot(epochs, history['val_auth'], color='green', linestyle='--', label='Authentic')
        if 'val_forg' in history: 
            axes[1, 0].plot(epochs, history['val_forg'], color='red', linestyle='--', label='Forged')
            
        axes[1, 0].set_title('Validation OF1 Score')
        axes[1, 0].set_xlabel('Epoch')
        axes[1, 0].legend()
        axes[1, 0].grid(True, alpha=0.3)
        
        # Annotate Max
        best_idx = np.argmax(history['val_of1'])
        best_val = history['val_of1'][best_idx]
        best_ep = epochs[best_idx]
        axes[1, 0].annotate(f'Best: {best_val:.4f}', (best_ep, best_val), xytext=(10, -20), 
                           textcoords='offset points', arrowprops=dict(arrowstyle="->"))

    # 4. Learning Rate
    if 'lr' in history:
        axes[1, 1].plot(epochs, history['lr'], color='orange')
        axes[1, 1].set_title('Learning Rate')
        axes[1, 1].set_xlabel('Epoch')
        axes[1, 1].set_yscale('log')
        axes[1, 1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(output_path, dpi=100)
    plt.close()
    
    print("✅ Graphs generated.")
    display(Image(output_path))

import numpy as np
plot_history()

❌ No history file found at /kaggle/working/outputs/training_history.json


In [ ]:
# import torch
# import matplotlib.pyplot as plt
# import numpy as np
# import cv2
# from torch.utils.data import DataLoader
# from config import Config
# from dataset import ForgeryDataset, get_train_transform

# def debug_dataloader_instances():
#     print("🔍 Starting Multi-Instance Mask Debugger...")
    
#     # 1. Config Setup
#     conf = Config()
#     conf.SMOKE_TEST = True
#     conf.SMOKE_TEST_SAMPLES = 8  # Keep it small for debugging
    
#     print(f"⚙️  Config: Input Size {conf.INPUT_SIZE}, Max Instances {conf.NUM_INSTANCES}")

#     # 2. Initialize Dataset
#     # We use get_train_transform to ensure augmentations (like flips) 
#     # are applied consistently to both images and all mask instances.
#     ds = ForgeryDataset(
#         authentic_dir=conf.TRAIN_AUTHENTIC_DIR,
#         forged_dir=conf.TRAIN_FORGED_DIR,
#         mask_dir=conf.TRAIN_MASK_DIR,
#         transform=get_train_transform(conf),
#         config=conf
#     )
    
#     # 3. Load Batch
#     batch_size = 4
#     loader = DataLoader(ds, batch_size=batch_size, shuffle=True)
    
#     try:
#         batch = next(iter(loader))
#     except Exception as e:
#         print(f"❌ Error loading batch: {e}")
#         return

#     inputs = batch['input']           # Shape: (B, 5, H, W)
#     masks = batch['masks']            # Shape: (B, NUM_INSTANCES, H, W)
#     is_forged = batch['is_forged']
#     num_instances = batch['num_instances']
#     image_names = batch['image_name']
    
#     # 4. Visualize
#     # We iterate through the batch items
#     for i in range(min(batch_size, len(inputs))):
        
#         # Determine number of subplots: 1 (Image) + NUM_INSTANCES (Masks)
#         n_cols = 1 + conf.NUM_INSTANCES
#         fig, axes = plt.subplots(1, n_cols, figsize=(3 * n_cols, 3.5))
        
#         # --- 1. Input Image (RGB only) ---
#         # Inputs are (C, H, W). Channels 0,1,2 are RGB. 3,4 are X,Y (ignored here).
#         img_tensor = inputs[i, :3] 
#         img = img_tensor.permute(1, 2, 0).numpy()
#         img = np.clip(img, 0, 1) # Ensure valid range for plotting
        
#         # Title info
#         status = "FORGED" if is_forged[i] > 0.5 else "AUTHENTIC"
#         n_inst = num_instances[i].item()
        
#         axes[0].imshow(img)
#         axes[0].set_title(f"{status}\n{image_names[i]}\nFound {n_inst} masks", fontsize=9, color='red' if n_inst>0 else 'green')
#         axes[0].axis('off')
        
#         # --- 2. Mask Instances ---
#         # Masks tensor shape: (NUM_INSTANCES, H, W)
#         instance_masks = masks[i]
        
#         for idx in range(conf.NUM_INSTANCES):
#             ax = axes[idx + 1]
            
#             # Get specific instance mask
#             m = instance_masks[idx].numpy()
            
#             # Check if this mask instance has content
#             has_content = m.max() > 0
            
#             # visual styling
#             cmap = 'hot' if has_content else 'gray'
#             title_color = 'red' if has_content else 'black'
#             title_text = f"Instance {idx+1}" + (" (Active)" if has_content else "")
            
#             ax.imshow(m, cmap=cmap, vmin=0, vmax=1)
#             ax.set_title(title_text, fontsize=9, color=title_color)
#             ax.axis('off')
            
#             # Optional: Add a border to active masks for visibility
#             if has_content:
#                 for spine in ax.spines.values():
#                     spine.set_edgecolor('red')
#                     spine.set_linewidth(2)

#         plt.tight_layout()
#         plt.show()

# if __name__ == "__main__":
#     debug_dataloader_instances()